In [5]:
import wfdb
import numpy as np
import os
import matplotlib.pyplot as plt
from collections import Counter

BASE_DIR = os.path.expanduser('~/Documents/ppg-arrhythmia-classifier')
DATA_DIR = f'{BASE_DIR}/data/mitdb'

# Window Configuration
WINDOW_SIZE = 180

# Beat labels
BEAT_LABELS = {
    'N': 'Normal',
    'V': 'Ventricular',
    'A': 'Atrial',
    'L': 'Left Bundle Branch Block',
    'R': 'Right Bundle Branch Block'
}

# All patient records
ALL_RECORDS = [
    '100','101','102','103','104','105','106','107',
    '108','109','111','112','113','114','115','116',
    '117','118','119','121','122','123','124','200',
    '201','202','203','205','207','208','209','210',
    '212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

# Verify path exists
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {os.path.exists(DATA_DIR)}")

Data directory: /Users/ryanjoseph/Documents/ppg-arrhythmia-classifier/data/mitdb
Directory exists: True


In [6]:
def extract_beats(record_name):
    """
    For a given patient record:
    - Load the ECG signal and beat annotations
    - Extract a window around each R-peak
    - Return segments and their labels
    """
    segments = []
    labels   = []

    try:
        record     = wfdb.rdrecord(f'{DATA_DIR}/{record_name}')
        annotation = wfdb.rdann(f'{DATA_DIR}/{record_name}', 'atr')

        signal = record.p_signal[:, 0]

        for i, (sample, symbol) in enumerate(
                zip(annotation.sample, annotation.symbol)):

            if symbol not in BEAT_LABELS:
                continue

            start = sample - WINDOW_SIZE
            end   = sample + WINDOW_SIZE

            if start < 0 or end > len(signal):
                continue

            segment = signal[start:end]
            segment = (segment - np.mean(segment)) / (np.std(segment) + 1e-8)

            segments.append(segment)
            labels.append(symbol)

    except Exception as e:
        print(f"Skipping record {record_name}: {e}")

    return segments, labels

print("extract_beats function ready")

extract_beats function ready


In [9]:
print("Extracting beats from all 48 patient records...")

all_segments = []
all_labels   = []

for rec in ALL_RECORDS:
    segs, labs = extract_beats(rec)
    all_segments.extend(segs)
    all_labels.extend(labs)
    print(f"  Record {rec}: {len(segs)} beats extracted")

X = np.array(all_segments)
y = np.array(all_labels)

print(f"\nDone!")
print(f"Total segments: {X.shape[0]}")
print(f"Segment length: {X.shape[1]} samples")
print(f"\nBeat type distribution:")
for label, count in Counter(y).most_common():
    print(f"  {label} ({BEAT_LABELS[label]}): {count} beats")

Extracting beats from all 48 patient records...
  Record 100: 2271 beats extracted
  Record 101: 1862 beats extracted
  Record 102: 103 beats extracted
  Record 103: 2083 beats extracted
  Record 104: 165 beats extracted
  Record 105: 2567 beats extracted
  Record 106: 2027 beats extracted
  Record 107: 59 beats extracted
  Record 108: 1759 beats extracted
  Record 109: 2528 beats extracted
  Record 111: 2124 beats extracted
  Record 112: 2537 beats extracted
  Record 113: 1787 beats extracted
  Record 114: 1873 beats extracted
  Record 115: 1951 beats extracted
  Record 116: 2411 beats extracted
  Record 117: 1534 beats extracted
  Record 118: 2277 beats extracted
  Record 119: 1987 beats extracted
  Record 121: 1861 beats extracted
  Record 122: 2474 beats extracted
  Record 123: 1517 beats extracted
  Record 124: 1579 beats extracted
  Record 200: 2598 beats extracted
  Record 201: 1852 beats extracted
  Record 202: 2115 beats extracted
  Record 203: 2972 beats extracted
  Record 20

In [10]:
PROCESSED_DIR = f'{BASE_DIR}/data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

np.save(f'{PROCESSED_DIR}/X_segments.npy', X)
np.save(f'{PROCESSED_DIR}/y_labels.npy', y)

print("Data saved successfully")
print(f"   X_segments.npy — shape: {X.shape}")
print(f"   y_labels.npy   — shape: {y.shape}")

Data saved successfully
   X_segments.npy — shape: (100012, 360)
   y_labels.npy   — shape: (100012,)
